# 0502 · 01 · Streaming tables con SQL y Auto Loader

En esta demostración vamos a crear una **streaming table** que ingiere de forma incremental los ficheros de un volumen usando Auto Loader desde SQL.

Cuando creas una streaming table con la sentencia `CREATE OR REFRESH STREAMING TABLE`, ni la carga inicial ni los refrescos posteriores consumen el cómputo del SQL warehouse o del notebook. Las streaming tables se apoyan en **pipelines serverless** tanto para crearse como para refrescarse: el sistema crea y gestiona automáticamente un pipeline serverless dedicado para cada streaming table.

## Objetivos de aprendizaje

Al terminar esta lección serás capaz de:

- Crear streaming tables en Databricks SQL para la ingesta incremental de datos.
- Refrescar streaming tables con la sentencia `REFRESH`.

## Antes de empezar

1. Ejecuta **`00_preparar_entorno`** (Run all). Deja el volumen `/Volumes/lab0502/upload/upload` con un único fichero, `000.csv`.
2. Conecta este notebook a **Serverless**.

## Recomendación

`CREATE STREAMING TABLE` es la alternativa recomendada al comando heredado `COPY INTO` para la ingesta incremental desde almacenamiento de objetos en la nube. Databricks recomienda usar streaming tables para ingerir datos con Databricks SQL.

Una streaming table es una tabla registrada en Unity Catalog con soporte adicional para el procesamiento de datos en streaming o incremental. Para cada streaming table se crea automáticamente un pipeline. Puedes usar streaming tables para la carga incremental de datos desde Kafka y desde almacenamiento de objetos en la nube.

Todas las tablas de este laboratorio se crean en el schema `lab0502.streaming`:

In [ ]:
%sql
USE CATALOG lab0502;
USE SCHEMA streaming;

SELECT current_catalog() AS catalogo, current_schema() AS schema;

## C. Crear streaming tables para procesamiento incremental

1. Explora el volumen `/Volumes/lab0502/upload/upload` y confirma que contiene un único fichero CSV:

   a. Selecciona el icono de **Catalog** en la barra de la izquierda.

   b. Despliega el catálogo **lab0502**.

   c. Despliega el schema **upload**.

   d. Despliega **Volumes**.

   e. Despliega el volumen **upload**.

   f. Confirma que contiene un único fichero CSV llamado **000.csv**.

   También puedes comprobarlo con código:

In [ ]:
%sql
LIST '/Volumes/lab0502/upload/upload';

2. Ejecuta la consulta de abajo para ver los datos del fichero o ficheros CSV de tu almacenamiento. Fíjate en que se devuelven en forma de tabla y en que el fichero contiene **3.149 filas**.

In [ ]:
%sql
-- Vista previa del CSV del volumen
SELECT *
FROM read_files(
  '/Volumes/lab0502/upload/upload',
  format => 'CSV',
  sep => '|',
  header => true
);

### Crear una STREAMING TABLE con Databricks SQL

3. El objetivo es crear un pipeline incremental que ingiera solo los ficheros nuevos, en lugar de usar la ingesta batch tradicional. Para ello usarás [streaming tables en Databricks SQL](https://docs.databricks.com/aws/en/ldp/dbsql/streaming) (Auto Loader).

   - El código SQL de abajo crea una streaming table que se programará para ingerir de forma incremental solo los datos nuevos **cada semana**.
   - Para cada streaming table se crea automáticamente un pipeline. Puedes usar streaming tables para la carga incremental de datos desde Kafka y desde almacenamiento de objetos en la nube.

   **NOTA:** la ingesta batch incremental detecta automáticamente los registros nuevos del origen e ignora los que ya se ingirieron. Así se reduce la cantidad de datos procesados y los trabajos de ingesta son más rápidos y aprovechan mejor los recursos de cómputo.

   La celda tarda alrededor de **un minuto**: tiene que crear y lanzar el pipeline serverless.

In [ ]:
%sql
-- Crear la streaming table
CREATE OR REFRESH STREAMING TABLE sql_csv_autoloader
SCHEDULE EVERY 1 WEEK     -- Programar el refresco es opcional
AS
SELECT *
FROM STREAM read_files(
  '/Volumes/lab0502/upload/upload',
  format => 'CSV',
  sep => '|',
  header => true
);

4. Localiza la streaming table en el catálogo:

   a. Selecciona el icono de **Catalog** en la barra de la izquierda.

   b. Despliega el catálogo **lab0502**.

   c. Despliega el schema **streaming**.

   d. Despliega **Tables**.

   e. Busca la tabla **sql_csv_autoloader**. Fíjate en que el icono de una streaming table es ligeramente distinto del de una tabla Delta tradicional: es una tabla con unas ondas, que indican que es streaming.

5. Ejecuta la celda de abajo para ver la streaming table. Confirma que el resultado tiene **3.149 filas**.

In [ ]:
%sql
-- Ver la streaming table
SELECT *
FROM sql_csv_autoloader;

6. Describe la STREAMING TABLE y revisa el resultado. Fíjate en lo siguiente:

   - En **Detailed Table Information**, estas filas:
     - **View Text**: la consulta que creó la tabla.
     - **Type**: indica que es una `STREAMING TABLE`.
     - **Provider**: indica que es una tabla Delta.
   - En **Refresh Information** ves los detalles de los refrescos. Por ejemplo (los valores y alguna fila pueden variar en tu workspace):

     | col_name | data_type |
     |---|---|
     | Last Refreshed | `<fecha y hora>` |
     | Last Refresh Type | INCREMENTAL / FULL |
     | Latest Refresh Status | Succeeded |
     | Latest Refresh | `<enlace a la actualización del pipeline>` |
     | Refresh Schedule | EVERY 1 WEEKS |

In [ ]:
%sql
DESCRIBE TABLE EXTENDED sql_csv_autoloader;

7. La sentencia `DESCRIBE HISTORY` muestra la lista detallada de cambios, versiones y metadatos asociados a una tabla Delta, incluidas actualizaciones, borrados y cambios de esquema.

   Ejecuta la celda de abajo y revisa el resultado. Fíjate en lo siguiente:

   - En la columna **operation** verás que una streaming table realiza tres operaciones: **CREATE TABLE**, **DLT SETUP** y **STREAMING UPDATE**.
   - Desplázate a la derecha hasta la columna **operationMetrics**. En la fila 1 (versión 2 de la tabla), la métrica `numOutputRows` vale **3149**: se añadieron 3.149 filas a la tabla **sql_csv_autoloader**.

In [ ]:
%sql
-- Historial de la streaming table
DESCRIBE HISTORY sql_csv_autoloader;

8. Añade otro fichero a tu almacenamiento, `/Volumes/lab0502/upload/upload`. La celda de abajo copia **001.csv** desde la carpeta `datasets/` del repositorio al volumen.

   > **Alternativa manual:** descarga `datasets/sales-csv/001.csv` del repositorio a tu ordenador y súbelo desde el explorador del catálogo. Haz clic derecho sobre el volumen **upload**, elige **Upload to volume** y selecciona el fichero. Según la configuración de seguridad de tu portátil, puede que no te deje descargar ficheros; en ese caso usa la celda.

In [ ]:
import os

# La carpeta de este notebook es la del laboratorio dentro de tu Git folder;
# los datasets viven un nivel más arriba, en la raíz del repositorio.
RUTA_LAB = os.getcwd()
if not RUTA_LAB.startswith("/Workspace"):
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    RUTA_LAB = "/Workspace" + os.path.dirname(ctx.notebookPath().get())
RUTA_DATASETS = f"{os.path.dirname(RUTA_LAB)}/datasets/sales-csv"

import shutil

shutil.copyfile(f"{RUTA_DATASETS}/001.csv", "/Volumes/lab0502/upload/upload/001.csv")
print("copiado 001.csv")

   Confirma que el volumen **upload** contiene ahora dos ficheros CSV (**000.csv** y **001.csv**).

In [ ]:
%sql
LIST '/Volumes/lab0502/upload/upload';

9. Refresca a mano la STREAMING TABLE con `REFRESH STREAMING TABLE <nombre-tabla>`. El refresco solo ingiere el fichero nuevo (**001.csv**), no los que ya se procesaron. Tarda alrededor de un minuto.

   **NOTA:** también puedes refrescarla desde la opción **Refresh** del explorador del catálogo, o dejar que lo haga la programación (`EVERY 1 WEEK`).

In [ ]:
%sql
REFRESH STREAMING TABLE sql_csv_autoloader;

10. Ejecuta la celda de abajo para ver los datos de la tabla **sql_csv_autoloader**. Fíjate en que ahora tiene **6.081 filas** (3.149 de `000.csv` más 2.932 de `001.csv`).

In [ ]:
%sql
-- Ver la streaming table
SELECT *
FROM sql_csv_autoloader;

11. Revisa el historial de la tabla **sql_csv_autoloader**. Fíjate en lo siguiente:

    - La versión 3 de la streaming table incluye otro **STREAMING UPDATE**.
    - Despliega la columna **operationMetrics**: solo se ingirieron de forma incremental **2.932 filas**, las del fichero nuevo `001.csv`. Las filas de `000.csv` no se volvieron a procesar.

In [ ]:
%sql
-- Historial de la streaming table
DESCRIBE HISTORY sql_csv_autoloader;

12. Borra la streaming table.

In [ ]:
%sql
DROP TABLE IF EXISTS sql_csv_autoloader;

## Recursos adicionales

- [Streaming tables (documentación)](https://docs.databricks.com/aws/en/ldp/concepts/streaming-tables)
- [Sintaxis de CREATE STREAMING TABLE](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-streaming-table)
- [Usar streaming tables en Databricks SQL](https://docs.databricks.com/aws/en/ldp/dbsql/streaming)
- [REFRESH (MATERIALIZED VIEW o STREAMING TABLE)](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-refresh-full)
- [COPY INTO (heredado)](https://docs.databricks.com/aws/en/sql/language-manual/delta-copy-into)

---

# Apéndice · Auto Loader con Python

### Material extra, no forma parte de la sesión en directo.

En esta demostración vas a ejecutar Auto Loader en Python para la ingesta incremental. Usarás el formato `cloudFiles` con Structured Streaming para ingerir los datos de forma incremental.

**Compute:** todo funciona en **Serverless**. Por eso el stream usa `trigger(availableNow=True)`: el `trigger(once=True)` del tutorial original está deprecado desde Spark 3.4 y **serverless no lo admite**. Los dos procesan lo pendiente y se paran; `availableNow` puede hacerlo en varios lotes en lugar de uno.

Antes de empezar, vuelve al estado inicial: la celda de abajo ejecuta `00_preparar_entorno`, que deja en el volumen **solo `000.csv`** y vacía el checkpoint.

In [ ]:
%run ./00_preparar_entorno

1. Lista los ficheros del volumen **upload**. Solo debe aparecer `000.csv`.

In [ ]:
spark.sql('LIST "/Volumes/lab0502/upload/upload"').display()

[¿Qué es Auto Loader?](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/)

[Usar Auto Loader con Unity Catalog](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/unity-catalog)

2. Ejecuta la celda de abajo para ingerir los CSV de forma incremental con Auto Loader en Python y escribirlos en la tabla **python_csv_autoloader**.

In [ ]:
## Crear un volumen para guardar los ficheros de checkpoint de Auto Loader
spark.sql('CREATE VOLUME IF NOT EXISTS lab0502.streaming.checkpoints')

## Apuntar el checkpoint a una carpeta de ese volumen
checkpoint_file_location = '/Volumes/lab0502/streaming/checkpoints/python_csv_autoloader'

## Ingerir de forma incremental (en streaming) con Auto Loader
(spark
 .readStream
   .format("cloudFiles")
   .option("cloudFiles.format", "csv")
   .option("header", "true")
   .option("sep", "|")
   .option("inferSchema", "true")   # Auto Loader la ignora: ver la nota del paso 3
   .option("cloudFiles.schemaLocation", f"{checkpoint_file_location}")
   .load("/Volumes/lab0502/upload/upload/")
 .writeStream
   .option("checkpointLocation", f"{checkpoint_file_location}")
   .trigger(availableNow=True)      # en el original: .trigger(once=True), no admitido en serverless
   .toTable("lab0502.streaming.python_csv_autoloader")
 .awaitTermination()                # espera a que termine antes de pasar a la siguiente celda
)

3. Consulta la tabla **python_csv_autoloader**.

In [ ]:
%sql
SELECT *
FROM lab0502.streaming.python_csv_autoloader;

   **NOTA:** fíjate en que todas las columnas se han ingerido como `STRING`, aunque la celda anterior pide `inferSchema`. Auto Loader **no usa la opción `inferSchema`**: en CSV y JSON infiere todas las columnas como texto. Para que infiera los tipos hay que añadir `.option("cloudFiles.inferColumnTypes", "true")`.

4. Ejecuta la celda de abajo para añadir un fichero nuevo, **001.csv**, al volumen **upload**.

In [ ]:
import os

# La carpeta de este notebook es la del laboratorio dentro de tu Git folder;
# los datasets viven un nivel más arriba, en la raíz del repositorio.
RUTA_LAB = os.getcwd()
if not RUTA_LAB.startswith("/Workspace"):
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    RUTA_LAB = "/Workspace" + os.path.dirname(ctx.notebookPath().get())
RUTA_DATASETS = f"{os.path.dirname(RUTA_LAB)}/datasets/sales-csv"

import shutil

shutil.copyfile(f"{RUTA_DATASETS}/001.csv", "/Volumes/lab0502/upload/upload/001.csv")
print("copiado 001.csv")

Confirma que el volumen contiene 2 ficheros CSV.

In [ ]:
spark.sql('LIST "/Volumes/lab0502/upload/upload"').display()

5. Vuelve a ejecutar el mismo stream de Auto Loader. Como el checkpoint recuerda qué ficheros ya se procesaron, solo se ingiere el nuevo (**001.csv**).

In [ ]:
checkpoint_file_location = '/Volumes/lab0502/streaming/checkpoints/python_csv_autoloader'

(spark
 .readStream
   .format("cloudFiles")
   .option("cloudFiles.format", "csv")
   .option("header", "true")
   .option("sep", "|")
   .option("inferSchema", "true")
   .option("cloudFiles.schemaLocation", f"{checkpoint_file_location}")
   .load("/Volumes/lab0502/upload/upload/")
 .writeStream
   .option("checkpointLocation", f"{checkpoint_file_location}")
   .trigger(availableNow=True)
   .toTable("lab0502.streaming.python_csv_autoloader")
 .awaitTermination()
)

6. Confirma que la tabla tiene ahora **6.081 filas**.

In [ ]:
%sql
SELECT count(*) AS filas
FROM lab0502.streaming.python_csv_autoloader;

7. Revisa el historial de la tabla **python_csv_autoloader**. Fíjate en los dos **STREAMING UPDATE**. En la columna **operationMetrics** verás cuántas filas se ingirieron en cada uno (3.149 y 2.932): solo se ingieren los ficheros nuevos.

In [ ]:
%sql
DESCRIBE HISTORY lab0502.streaming.python_csv_autoloader;

8. Borra la tabla **python_csv_autoloader**.

In [ ]:
%sql
DROP TABLE IF EXISTS lab0502.streaming.python_csv_autoloader;